# BÁO CÁO ĐỒ ÁN MÔN HỌC
## Dự báo doanh số bán hàng theo chuỗi thời gian bằng mô hình kết hợp (Ensemble) — bản chạy LOCAL

---

**Sinh viên thực hiện:** *(điền họ tên)*  
**Mã số sinh viên:** *(điền MSSV)*  
**Lớp / Môn học:** *(điền tên môn học)*  
**Giảng viên hướng dẫn:** *(điền tên giảng viên)*  
**Thời gian:** Tháng 06/2026

---

## Tóm tắt

Đồ án giải quyết bài toán **dự báo doanh số bán hàng** trên bộ dữ liệu *"Store Sales -
Time Series Forecasting"* (Kaggle): dự báo lượng bán theo ngày cho 1782 chuỗi
(cửa hàng × nhóm hàng). Thay vì dùng một mô hình đơn lẻ, đồ án **tái lập và phân tích** một **mô hình
kết hợp (ensemble)** (theo repo tham khảo `mojaevr/store-sales-forecasting`) gồm 4 thành phần đa dạng — họ cây tăng cường gradient, mô hình nền
tảng Chronos-2, LightGBM điều chuẩn, và mạng nơ-ron TSMixer — rồi **trộn** chúng theo
phương pháp *cực tiểu phương sai (minimum-variance)*. Chỉ số đánh giá chính là **RMSLE**.

Kết quả: mỗi mô hình huấn luyện **một lần** ra một file dự đoán `submission_*.csv`; **RMSLE
riêng (σ) của từng nhánh do nhóm tự đo bằng cách NỘP file đó lên Kaggle** (nhãn tập test
do Kaggle giữ). Từ các σ này, code tính **trọng số trộn cực tiểu phương sai** và **RMSLE blend
ước lượng**; **RMSLE cuối cùng chính thức** là điểm Kaggle khi nộp file blend.

## Mục lục

1. **Chương 1 — Giới thiệu:** bài toán, mục tiêu, phạm vi.
2. **Chương 2 — Cài đặt & dữ liệu:** môi trường, cấu hình chạy, bộ dữ liệu, thư viện.
3. **Chương 3 — Phương pháp:** kiến trúc ensemble, kỹ thuật trộn, bốn mô hình thành phần.
4. **Chương 4 — Thực nghiệm & kết quả:** RMSLE từng mô hình, trọng số trộn, đối chiếu, thời gian huấn luyện.
5. **Chương 5 — Kết luận & hướng phát triển.**
6. **Tài liệu tham khảo.**

> **Cách chạy:** notebook nằm ở **thư mục gốc dự án** (cạnh `model/`, `data/`,
> `submissions/`). Mặc định `RUN_LEGS=False`: bỏ qua huấn luyện,
> chỉ **lắp ráp lại** kết quả từ các file `submission_*.csv` có sẵn rồi chạy quy trình
> ở Chương 4. Nếu muốn tự train (`RUN_LEGS=True`) cần cài darts/lightgbm/torch trong venv
> và runtime Chronos-2 riêng — xem Chương 2; trên máy không GPU việc này rất chậm/không khả thi.

## Chương 1 — Giới thiệu

### 1.1 Đặt vấn đề

Dự báo doanh số là bài toán quan trọng trong bán lẻ: ước lượng đúng lượng hàng bán ra
giúp tối ưu tồn kho, nhân sự và khuyến mãi. Bộ dữ liệu *Store Sales* (chuỗi siêu thị
Favorita, Ecuador) cung cấp lịch sử bán hàng theo ngày của nhiều cửa hàng và nhóm hàng,
kèm các yếu tố ảnh hưởng như khuyến mãi, ngày lễ và giá dầu. Nhiệm vụ: **dự báo lượng
bán 16 ngày kế tiếp** cho từng cặp (cửa hàng, nhóm hàng).

Đặc thù: dữ liệu là **chuỗi thời gian nhiều biến, quy mô lớn** (1782 chuỗi), có tính mùa
vụ, chịu ảnh hưởng của sự kiện (lễ, khuyến mãi) và nhiễu lớn ở các giá trị nhỏ — rất phù
hợp để so sánh nhiều hướng mô hình hóa.

### 1.2 Mục tiêu

- Xây dựng và so sánh **bốn hướng mô hình** khác nhau cho cùng một bài toán.
- Áp dụng **kết hợp mô hình (ensemble)** theo *cực tiểu phương sai* để đạt sai số thấp
  hơn mọi mô hình đơn lẻ.
- Đánh giá bằng **RMSLE** (chỉ số chính) và báo cáo **thời gian huấn luyện** từng mô hình.

## Chương 2 — Cài đặt & dữ liệu

Chương này mô tả môi trường thực nghiệm, cách cấu hình chạy, bộ dữ liệu và thư viện.

### 2.1 Môi trường & đường dẫn dự án

Nạp các thư viện rồi xác định đường dẫn dự án từ **thư mục hiện hành** (`model/`,
`submissions/`, `data/`) — bản local không mount Google Drive. Thư mục `model/` được thêm
vào `sys.path` để import các script và thư viện `blend`. Hãy mở notebook này ngay tại thư
mục gốc dự án (hoặc sửa `REPO` trong ô bên dưới cho đúng).

In [1]:
import os
import sys
import time
import shlex
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
IN_COLAB = False

REPO = Path.cwd()
if not (REPO / "model").exists() and (REPO.parent / "model").exists():
    REPO = REPO.parent

SRC = REPO / "model"
SUBMISSIONS = REPO / "submissions"
DATA = REPO / "data"
SUBMISSIONS.mkdir(parents=True, exist_ok=True)

assert SRC.exists(), f"Script folder not found: {SRC}. Open the notebook at the project root or set REPO."
assert DATA.exists(), f"Data folder not found: {DATA}."

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import blend

print(f"REPO         = {REPO}")
print(f"SRC          = {SRC}")
print(f"DATA         = {DATA}")
print(f"SUBMISSIONS  = {SUBMISSIONS}")

REPO         = /Users/ngohoangkhactuong/Downloads/Colab Notebooks
SRC          = /Users/ngohoangkhactuong/Downloads/Colab Notebooks/model
DATA         = /Users/ngohoangkhactuong/Downloads/Colab Notebooks/data
SUBMISSIONS  = /Users/ngohoangkhactuong/Downloads/Colab Notebooks/submissions


### 2.2 Cấu hình chạy

`RUN_LEGS` là công tắc chính. Khi để **False** (mặc định), hàm `run()` không làm gì:
notebook chỉ *lắp ráp lại* kết quả cuối từ các file CSV đã có sẵn. Đặt **True** nếu muốn
huấn luyện lại từ đầu (cần GPU và dữ liệu). `run()` là hàm bọc mỏng để chạy một lệnh
shell từ thư mục gốc dự án.

**Ý nghĩa các dòng log ở mỗi ô huấn luyện (Chương 3):**
- `[bỏ qua] <mô hình> (RUN_LEGS=False)` — bỏ qua vì đang ở chế độ chỉ lắp ráp.
- `[đã có] <mô hình> -> … đã có trong submissions/` — đã có sẵn CSV nên không huấn luyện lại.
- `[chạy] <mô hình> (<lệnh>)` rồi `[xong] <mô hình> trong 3m 12.0s` — đang huấn luyện, kèm thời gian.

In [ ]:
RUN_LEGS = True         # LOCAL: set True to train legs yourself; False = only reassemble from existing CSVs
FORCE_RETRAIN = True    # True = retrain a model even if its result CSV already exists
TIMINGS = {}            # model label -> elapsed time (seconds)


def _label_for(cmd, env):
    """Short, readable name for a training command (used in logs and the timing table)."""
    if env and "OUT_NAME" in env:
        return env["OUT_NAME"].replace("submission_", "").replace(".csv", "")
    for tok in shlex.split(cmd):
        if tok.endswith(".py"):
            return Path(tok).stem
    return cmd


def run(cmd, env=None, cwd=None, produces=None):
    """Run a training command. Two cases are skipped:

      * RUN_LEGS = False          -> skip every model (reassemble-only mode).
      * `produces` already exists -> skip this model; its CSV is already in
        submissions/ (set FORCE_RETRAIN=True to rebuild).

    `produces` is the result filename in submissions/. If omitted it defaults to
    env["OUT_NAME"]. Elapsed time is recorded in TIMINGS for the timing table (section 4.5).
    """
    cwd = cwd or REPO
    label = _label_for(cmd, env)
    produces = produces or (env or {}).get("OUT_NAME")
    if env:
        print("  params:", " ".join(f"{k}={v}" for k, v in env.items()))

    if not RUN_LEGS:
        print(f"  [skip]   {label}  (RUN_LEGS=False)")
        return
    if produces and (SUBMISSIONS / produces).exists() and not FORCE_RETRAIN:
        print(f"  [exists] {label}  ->  {produces} already in submissions/, skipping "
              f"(set FORCE_RETRAIN=True to rebuild)")
        return

    print(f"  [run]    {label}  ({cmd})")
    full_env = {**os.environ, **(env or {})}
    argv = shlex.split(cmd)
    if argv and argv[0] == "python":
        argv[0] = sys.executable
    t0 = time.time()
    subprocess.run(argv, env=full_env, cwd=str(cwd), check=True)
    elapsed = time.time() - t0
    TIMINGS[label] = elapsed
    mins, secs = divmod(elapsed, 60)
    print(f"  [done]   {label}  in {int(mins)}m {secs:04.1f}s")


RUN_LEGS, FORCE_RETRAIN

(False, True)

### 2.3 Bộ dữ liệu

Bộ dữ liệu gồm **6 file CSV** trong thư mục `data/`:

| File | Nội dung |
|------|----------|
| `train.csv` | Lịch sử bán hàng theo ngày (cửa hàng × nhóm hàng × ngày) + cờ khuyến mãi. |
| `test.csv` | Khung 16 ngày cần dự báo. |
| `stores.csv` | Thông tin cửa hàng (thành phố, loại, cụm). |
| `oil.csv` | Giá dầu theo ngày (yếu tố vĩ mô của Ecuador). |
| `holidays_events.csv` | Ngày lễ và sự kiện. |
| `transactions.csv` | Số giao dịch theo cửa hàng/ngày. |

Ô mã bên dưới kiểm tra đủ 6 file. (`sample_submission.csv` của Kaggle không dùng tới.)

In [ ]:
expected = ["train.csv", "test.csv", "stores.csv", "oil.csv",
            "holidays_events.csv", "transactions.csv"]

present = [f for f in expected if (DATA / f).exists()]
missing = [f for f in expected if not (DATA / f).exists()]

print(f"data folder: {DATA}")
print("present:", present)
if missing:
    raise FileNotFoundError(
        f"Missing data files in {DATA}: {missing}. "
        "Please place them into the data/ folder."
    )
print("All data files present.")
present

thư mục dữ liệu: /Users/ngohoangkhactuong/Downloads/Colab Notebooks/data
đang có: ['train.csv', 'test.csv', 'stores.csv', 'oil.csv', 'holidays_events.csv', 'transactions.csv']
Đã có đủ tất cả file dữ liệu.


['train.csv',
 'test.csv',
 'stores.csv',
 'oil.csv',
 'holidays_events.csv',
 'transactions.csv']

### 2.4 Thư viện cho việc huấn luyện

Các thư viện cần để huấn luyện: darts, LightGBM, XGBoost, CatBoost, scikit-learn. Bản local **không tự cài** ở chế độ lắp ráp. Ô **"Cài thư viện"** bên dưới chỉ chạy khi
`RUN_LEGS=True` — nó cài darts/torch/lightgbm/... vào đúng trình thông dịch đang chạy notebook.
Nhánh Chronos-2 cần runtime riêng (Python 3.11 + chronos-forecasting) và **không** cài ở đây — xem mục 3.4.

In [ ]:
# --- (Optional) Install libraries to TRAIN locally - runs only when RUN_LEGS=True ---
# Installs into the SAME interpreter running this notebook (sys.executable); legs invoked as
# "python ..." are remapped to sys.executable by run(), so they work immediately. The Chronos-2
# leg (section 3.4) is NOT installed here - it needs its own py3.11 venv (.venv_chronos2).
if RUN_LEGS:
    pkgs = ["darts", "torch", "lightgbm", "xgboost", "catboost",
            "scikit-learn", "tqdm", "pyarrow"]
    print("Installing (may take a few minutes):", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", *pkgs], check=True)

    import importlib
    for m in ["darts", "torch", "lightgbm", "xgboost", "catboost", "sklearn"]:
        try:
            importlib.import_module(m); print(f"  OK      {m}")
        except Exception:
            print(f"  MISSING {m} - check the install log above")
    print("Note: training darts/TSMixer on CPU is very slow; Chronos-2 needs its own .venv_chronos2.")
else:
    print("RUN_LEGS=False - skipping install (reassemble-only mode from existing CSVs).")

## Chương 3 — Phương pháp

### 3.1 Tổng quan: kết hợp mô hình theo cực tiểu phương sai

Ý tưởng cốt lõi: thay vì chọn một mô hình duy nhất, ta **huấn luyện nhiều mô hình đa
dạng** rồi **trộn** dự đoán của chúng. Kết quả cuối là blend của **4 mô hình** trong
không gian `log1p`; riêng nhánh "family" lại là tổ hợp 6 biến thể cây tăng cường gradient.

**Vì sao trộn theo *minimum-variance*?** Mỗi mô hình có một sai số riêng (RMSLE). Phương
pháp dựng lại ma trận hiệp phương sai của lỗi *chỉ dựa trên mức độ các dự đoán khác nhau*,
rồi chọn bộ trọng số làm **cực tiểu phương sai của lỗi tổng hợp**:

$$w = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^{\top}\Sigma^{-1}\mathbf{1}}$$

Hai mô hình càng khác nhau thì lỗi càng ít tương quan và trộn càng lợi; **trọng số được
phép âm** để một mô hình chủ động triệt tiêu lỗi mà các mô hình khác cùng mắc. Bộ trọng
số và RMSLE cụ thể được trình bày ở Chương 4.

Bốn mô hình thành phần (RMSLE riêng = điểm Kaggle khi nộp file của nhánh; trọng số tính ở mục 4.3):

| Mô hình | Ý tưởng chính |
|---------|---------------|
| family (mục 3.2) | 6 biến thể cây tăng cường gradient |
| lgbm-reg (mục 3.3) | LightGBM điều chuẩn mạnh |
| chronos2-cov (mục 3.4) | mô hình nền tảng Chronos-2 |
| tsmixer (mục 3.5) | mạng nơ-ron TSMixer |

> Các ô mã huấn luyện bên dưới chỉ thực thi khi `RUN_LEGS=True`; mỗi mô hình train **một lần**
> và chỉ sinh **dự đoán test** dưới dạng `submission_*.csv`. Không còn bước holdout/`*.parquet`:
> RMSLE riêng của mỗi nhánh lấy bằng cách **nộp file CSV đó lên Kaggle** (mục 4.2).

### 3.2 Mô hình 1/4 — Họ cây tăng cường gradient (6 biến thể)

Chạy `darts_lgbm_family.py` 6 lần; mỗi lần là một mô hình theo từng nhóm hàng, trung bình
trên 4 cấu hình độ trễ (lag = 56, 7, 365, 730), chọn qua biến môi trường:

- base: LightGBM mặc định.
- deeper: cây sâu hơn (DEPTH=8).
- xgb: thêm thành viên XGBoost (INCLUDE_XGB=1).
- subsampled: lấy mẫu dòng & cột để giảm phương sai.
- weighted: tăng trọng số cho dữ liệu gần đây (sqrt).
- cat_deep: chỉ CatBoost, cây sâu hơn.

6 file CSV này được `blend.build_family()` gộp lại thành 1 nhánh "family".

In [ ]:
run("python model/darts_lgbm_family.py",
    env={"OUT_NAME": "submission_darts_lgbm.csv"})

In [ ]:
run("python model/darts_lgbm_family.py",
    env={"DEPTH": "8", "OUT_NAME": "submission_darts_lgbm_deeper.csv"})

In [ ]:
run("python model/darts_lgbm_family.py",
    env={"INCLUDE_XGB": "1", "OUT_NAME": "submission_darts_xgb_lgbm.csv"})

In [ ]:
run("python model/darts_lgbm_family.py",
    env={"BAGGING_FRACTION": "0.8", "BAGGING_FREQ": "1", "FEATURE_FRACTION": "0.8",
         "OUT_NAME": "submission_darts_lgbm_subsampled_3seed.csv"})

In [ ]:
run("python model/darts_lgbm_family.py",
    env={"SAMPLE_WEIGHT_FLOOR": "1", "WEIGHT_SCHEME": "sqrt",
         "OUT_NAME": "submission_darts_lgbm_w.csv"})

In [ ]:
run("python model/darts_lgbm_family.py",
    env={"CAT_ONLY": "1", "DEPTH": "8", "OUT_NAME": "submission_darts_cat_deeper.csv"})

### 3.3 Mô hình 2/4 — LightGBM điều chuẩn mạnh (v8)

Một nhánh LightGBM điều chuẩn (regularization) mạnh, thêm các đặc trưng liên quan khuyến
mãi (cannibalization, tỉ lệ promo theo cửa hàng, promo theo thứ trong tuần). RMSLE riêng
tương đối cao (xem mục 4.2) nhưng *lỗi ít tương quan* với các nhánh khác nên blend vẫn gán
cho nó một trọng số dương nhỏ. Ghi ra `submission_v8_reg.csv`.

In [ ]:
run("python model/lgbm_regularized.py --suffix reg",
    produces="submission_v8_reg.csv")

### 3.4 Mô hình 3/4 — Chronos-2 kèm biến phụ trợ (covariates)

Mô hình nền tảng Chronos-2 của Amazon, chạy với các biến phụ trợ biết trước: `onpromotion`
và cờ ngày lễ biết trên toàn bộ horizon, còn giá dầu (oil) là biến chỉ biết trong quá khứ.
Nhánh này chạy trong môi trường riêng (`.venv_chronos2`: Python 3.11 + chronos-forecasting
2.2.2). Ghi ra `submission_chronos2_cov_promo.csv`.

In [ ]:
run(".venv_chronos2/bin/python model/chronos2_cov.py --suffix _promo",
    produces="submission_chronos2_cov_promo.csv")

### 3.5 Mô hình 4/4 — Mạng nơ-ron TSMixer

Một mô hình nơ-ron toàn cục (darts TSMixer) huấn luyện trên cả 1782 chuỗi (cửa hàng × nhóm
hàng) bằng GPU, dùng các biến khuyến mãi, giá dầu, lịch và ngày lễ. Script ghi vào `out/`,
nên ô thứ hai copy kết quả sang `submissions/` theo đúng tên blend mong đợi
(`submission_tsmixer_tuned.csv`).

In [ ]:
run("python model/neural_tsmixer.py --model tsmixer --epochs 30",
    env={"CUDA_VISIBLE_DEVICES": "0", "HID": "128", "FF": "128", "BLK": "8"},
    produces="submission_tsmixer_tuned.csv")

In [ ]:
run("cp out/submission_tsmixer.csv submissions/submission_tsmixer_tuned.csv",
    produces="submission_tsmixer_tuned.csv")

## Chương 4 — Thực nghiệm & kết quả

### 4.1 Kiểm tra dự đoán của các mô hình

Trước khi trộn, xác nhận mọi file CSV dự đoán của các nhánh đều có trong `submissions/`:
6 biến thể họ cây, cùng chronos2-cov, lgbm-reg và tsmixer.

In [6]:
needed = list(blend.FAMILY_FILES.values()) + [
    "submission_chronos2_cov_promo.csv",
    "submission_v8_reg.csv",
    "submission_tsmixer_tuned.csv",
]
missing = []
for f in needed:
    ok = (SUBMISSIONS / f).exists()
    print(f"  {'✓' if ok else '✗'}  {f}")
    if not ok:
        missing.append(f)

  ✓  submission_darts_lgbm.csv
  ✓  submission_darts_lgbm_deeper.csv
  ✓  submission_darts_xgb_lgbm.csv
  ✓  submission_darts_lgbm_subsampled_3seed.csv
  ✓  submission_darts_cat_deeper.csv
  ✓  submission_darts_lgbm_w.csv
  ✓  submission_chronos2_cov_promo.csv
  ✓  submission_v8_reg.csv
  ✓  submission_tsmixer_tuned.csv


### 4.2 Nộp bài & lấy điểm Kaggle (lấy σ cho từng nhánh)

RMSLE riêng (σ) của mỗi nhánh là **điểm leaderboard khi nộp file CSV của nhánh đó** — không
tự tính tại local vì nhãn tập test do Kaggle giữ. Hai cách nộp:

**Cách A — giao diện web:** mở
[trang nộp bài](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/submit),
chọn *Submit Predictions*, tải lên một file `submissions/submission_*.csv`, đợi chấm rồi đọc
cột *Score* (chính là RMSLE).

**Cách B — dòng lệnh `kaggle`** (cấu hình `~/.kaggle/kaggle.json` một lần):

    pip install kaggle
    kaggle competitions submit -c store-sales-time-series-forecasting \
        -f submissions/submission_v8_reg.csv -m "lgbm-reg"
    kaggle competitions submissions -c store-sales-time-series-forecasting   # xem điểm

Các file cần nộp để có đủ σ (Kaggle cho ~5 lượt/ngày):

| File nộp | σ điền vào `model/blend.py` |
|---|---|
| 6 file họ cây `submission_darts_*.csv` | `FAMILY_SIGMA` (6 thành viên) |
| `submission_family.csv` *(sinh ở mục 4.4)* | `LEG_SIGMA["family"]` |
| `submission_chronos2_cov_promo.csv` | `LEG_SIGMA["chronos2-cov"]` |
| `submission_v8_reg.csv` | `LEG_SIGMA["lgbm-reg"]` |
| `submission_tsmixer_tuned.csv` | `LEG_SIGMA["tsmixer"]` |
| `submission_fam_cov_v8_tsmTuned_4way.csv` *(mục 4.4)* | → **RMSLE cuối cùng** của đồ án |

Sau khi có điểm, **sửa các hằng số σ trong `model/blend.py`** (`FAMILY_SIGMA`, `LEG_SIGMA`)
rồi chạy lại ô dưới + mục 4.3/4.4. Số điền sẵn là **tham khảo của repo gốc** — thay bằng
điểm bạn tự nộp.

In [ ]:
# === Current sigma table (Kaggle score of each submission file) ===
# Edit FAMILY_SIGMA / LEG_SIGMA in model/blend.py, then rerun this cell to refresh.
import importlib
importlib.reload(blend)

rows = [(blend.FAMILY_FILES[m], blend.FAMILY_SIGMA[m]) for m in blend.FAMILY_FILES]
rows += [(blend.FAMILY_OUT_FILE, blend.LEG_SIGMA["family"])]
rows += [(blend.LEG_FILES[k], blend.LEG_SIGMA[k]) for k in blend.LEG_FILES]
display(pd.DataFrame(rows, columns=["submission file", "RMSLE Kaggle (sigma)"]))

,file nộp,RMSLE Kaggle (σ)
0,submission_darts_lgbm.csv,0.38345
1,submission_darts_lgbm_deeper.csv,0.38098
2,submission_darts_xgb_lgbm.csv,0.38231
3,submission_darts_lgbm_subsampled_3seed.csv,0.38381
4,submission_darts_cat_deeper.csv,0.38182
5,submission_darts_lgbm_w.csv,0.38344
6,submission_family.csv,0.37856
7,submission_chronos2_cov_promo.csv,0.40100
8,submission_v8_reg.csv,0.48297
9,submission_tsmixer_tuned.csv,0.38191


→ σ là điểm leaderboard của ĐÚNG file đó; thay số tham khảo bằng điểm bạn tự nộp.


### 4.3 Trọng số trộn & RMSLE blend (ước lượng)

Từ các σ (điểm Kaggle, mục 4.2) và mức chênh lệch giữa các dự đoán, `blend.build_fourway()`
dựng lại ma trận hiệp phương sai của lỗi rồi tính **trọng số trộn cực tiểu phương sai** và
**RMSLE blend ước lượng** — toàn bộ bằng công thức, không cần nhãn thật. Trọng số có thể âm
(mục 3.1). RMSLE *chính thức* của blend lấy ở mục 4.4 bằng cách nộp file blend lên Kaggle.

In [8]:
# === 4-leg blend — weights & estimated RMSLE from sigma (Kaggle scores) ===
# build_fourway() reads the sigmas in blend.LEG_SIGMA + each leg's test predictions, then
# computes minimum-variance weights and the blend RMSLE by FORMULA (no ground truth needed).
r = blend.build_fourway()

table = pd.DataFrame({
    "model":                r["legs"],
    "RMSLE (Kaggle sigma)": np.round(r["sigmas"], 5),   # per-leg standalone leaderboard score
    "weight":               np.round(r["weights"], 3),  # share in the blend (can be negative)
})
display(table)

i = int(np.argmin(r["sigmas"])); best_leg = r["legs"][i]; best = r["sigmas"][i]
print(f"best single model             : {best_leg} (RMSLE {best:.5f})")
print(f"blend RMSLE (formula estimate): {r['blend_rmsle_formula']:.5f}")
print(f"improvement over best single  : {(best - r['blend_rmsle_formula']) / best * 100:.1f}%")
print("The OFFICIAL final RMSLE is obtained only by SUBMITTING the blend file to Kaggle (section 4.4).")

,mô hình,RMSLE (Kaggle σ),trọng số
0,family,0.37856,0.505
1,chronos2-cov,0.40100,0.148
2,lgbm-reg,0.48297,0.010
3,tsmixer,0.38191,0.337


mô hình đơn tốt nhất            : family (RMSLE 0.37856)
blend — RMSLE ước lượng (CT)   : 0.37343
cải thiện so với nhánh tốt nhất : 1.4%
RMSLE CUỐI CÙNG chính thức chỉ có khi NỘP file blend lên Kaggle (mục 4.4).


> **Diễn giải kết quả mục 4.3**
>
> **Bảng:**
> - `mô hình` — 4 nhánh thành phần của blend.
> - `RMSLE (Kaggle σ)` — sai số *riêng* của nhánh = **điểm leaderboard khi nộp file của nhánh**
>   (mục 4.2). **Càng nhỏ càng tốt.**
> - `trọng số` — tỉ trọng trong blend (tổng ≈ 1), có thể **âm** để triệt tiêu lỗi chung.
>
> **Các dòng in bên dưới:**
> - `mô hình đơn tốt nhất` — nhánh có σ thấp nhất.
> - `blend — RMSLE ước lượng (CT)` — RMSLE blend theo công thức cực tiểu phương sai (ước lượng).
> - `cải thiện` — blend giảm bao nhiêu % so với nhánh đơn tốt nhất (theo ước lượng).
>
> **RMSLE cuối cùng chính thức** không tính ở đây — chỉ có khi nộp file blend lên Kaggle (mục 4.4).

> **Bảng nguồn số liệu**
>
> | Số liệu | Nguồn | Cách lấy |
> |---|---|---|
> | RMSLE riêng (σ) từng nhánh | **Kaggle leaderboard** | nộp file `submission_*.csv` của nhánh (mục 4.2) |
> | RMSLE 6 thành viên *family* | **Kaggle leaderboard** | nộp 6 file `submission_darts_*.csv` |
> | Trọng số trộn | **Code đồ án** | `blend.min_var_weights(σ, dự đoán)` từ σ + chênh lệch dự đoán |
> | RMSLE blend (ước lượng) | **Code đồ án** | công thức cực tiểu phương sai |
> | RMSLE blend (chính thức) | **Kaggle leaderboard** | nộp `submission_fam_cov_v8_tsmTuned_4way.csv` (mục 4.4) |
> | Dự đoán tập test (file nộp) | **Code đồ án** | áp trọng số lên dự đoán test của từng nhánh |
>
> Trọng số và RMSLE ước lượng **tính hoàn toàn từ σ + dự đoán** (không cần nhãn thật, không
> train thêm). Mỗi mô hình chỉ train **một lần** ra file CSV; σ và RMSLE cuối là điểm Kaggle.

### 4.4 Dựng file nộp cuối & lấy RMSLE chính thức

Chạy `build_ensemble.py`: nó dùng σ (điểm Kaggle) trong `blend.LEG_SIGMA` + dự đoán test để
tính trọng số rồi ghi **file nộp cuối** `submission_fam_cov_v8_tsmTuned_4way.csv` (kèm
`submission_family.csv` để nộp lấy σ cho nhánh *family*). Chạy lại với `--verify` để dựng
**hai lần** và khẳng định pipeline **tái lập được** (kết quả ổn định).

**Lấy RMSLE cuối cùng:** nộp `submission_fam_cov_v8_tsmTuned_4way.csv` lên Kaggle (mục 4.2) —
điểm leaderboard chính là RMSLE chính thức của đồ án (tham khảo: repo gốc ≈ 0.37418).

In [9]:
script = str(SRC / "build_ensemble.py")
out = subprocess.run([sys.executable, script], capture_output=True, text=True)
print(out.stdout, out.stderr)

verify = subprocess.run([sys.executable, script, "--verify"], capture_output=True, text=True)
print(verify.stdout, verify.stderr)
if verify.returncode == 0:
    print("Pipeline reproduces stably. Result file:", SUBMISSIONS / "submission_fam_cov_v8_tsmTuned_4way.csv")
else:
    print("[!] Build failed - missing leg CSV files (set RUN_LEGS=True and retrain in Chapter 3).")

weights family=0.5048 chronos2-cov=0.1476 lgbm-reg=0.0104 tsmixer=0.3372
math_LB = 0.37343  [reference actual LB 0.37418]
diff family<->tsmixer = 0.133 (family-level redundancy watch)
Wrote submission_fam_cov_v8_tsmTuned_4way.csv  (rows=28512, max sales=14682)
Wrote submission_family.csv  (rows=28512)
 
weights family=0.5048 chronos2-cov=0.1476 lgbm-reg=0.0104 tsmixer=0.3372
math_LB = 0.37343  [reference actual LB 0.37418]
diff family<->tsmixer = 0.133 (family-level redundancy watch)
VERIFY max|delta| = 1.819e-12  ->  EXACT MATCH
 
Pipeline tái lập ổn định. File kết quả: /Users/ngohoangkhactuong/Downloads/Colab Notebooks/submissions/submission_fam_cov_v8_tsmTuned_4way.csv


> **Diễn giải phần in ra của `build_ensemble.py` (mục 4.4)**
>
> - `weights family=… chronos2-cov=… …` — trọng số trộn của 4 nhánh (khớp bảng mục 4.3).
> - `math_LB = …` — RMSLE blend **ước lượng** theo công thức cực tiểu phương sai.
> - `diff family<->tsmixer` — mức khác biệt giữa hai dự đoán; đủ lớn ⇒ hai nhánh bổ sung tốt cho nhau.
> - `Wrote submission_fam_cov_v8_tsmTuned_4way.csv …` — đã ghi file nộp cuối.
> - `Wrote submission_family.csv …` — đã ghi file nhánh *family* (nộp để lấy σ của nó).
> - `VERIFY max|delta| ≈ 0 -> EXACT MATCH` — dựng lại hai lần cho cùng kết quả ⇒ tái lập được.
>
> RMSLE **chính thức** của blend = điểm Kaggle khi nộp file cuối (không in ở đây).

### 4.5 Thời gian huấn luyện từng mô hình

Thời gian huấn luyện (đồng hồ thực) của mỗi mô hình, lâu nhất xếp trước. Chỉ có số liệu khi
thực sự huấn luyện (`RUN_LEGS = True`); với mặc định `False` thì không có.

**Ý nghĩa cột:** `mô hình` — tên mô hình; `phút`/`giây` — thời gian huấn luyện. Dòng tổng kết
cho biết tổng thời gian, mô hình lâu nhất và nhanh nhất.

In [10]:
if TIMINGS:
    items = sorted(TIMINGS.items(), key=lambda kv: kv[1], reverse=True)
    table = pd.DataFrame(
        [{"model": k, "minutes": round(v / 60, 2), "seconds": round(v, 1)}
         for k, v in items]
    )
    display(table)

    total_minutes = table["minutes"].sum()
    print(f"trained {len(items)} models, total {total_minutes:.1f} min | "
          f"slowest {items[0][0]} ({items[0][1] / 60:.1f} min) | "
          f"fastest {items[-1][0]} ({items[-1][1] / 60:.1f} min)")
else:
    print("No training times yet - RUN_LEGS is False, so no model was trained.")
    print("Set RUN_LEGS = True and rerun the training cells to populate this.")

Chưa có thời gian train — RUN_LEGS đang là False nên không mô hình nào được train.
Đặt RUN_LEGS = True rồi chạy lại các ô train để có số liệu ở đây.


## Chương 5 — Kết luận & hướng phát triển

### 5.1 Kết luận

- Đồ án đã xây dựng pipeline dự báo doanh số theo hướng **kết hợp 4 mô hình đa dạng**.
- Kỹ thuật trộn *cực tiểu phương sai* cho **RMSLE thấp hơn mô hình đơn tốt nhất** (σ lấy từ
  điểm Kaggle; trọng số & RMSLE ước lượng ở mục 4.3) — minh chứng rằng kết hợp các mô hình
  *ít tương quan* giúp giảm sai số.
- Mô hình `lgbm-reg` tuy RMSLE riêng cao vẫn được giữ (trọng số dương nhỏ) nhờ
  lỗi *ít tương quan* với các nhánh khác — cho thấy **sự đa dạng** quan trọng không kém
  độ chính xác đơn lẻ.
- Pipeline cho **kết quả ổn định khi dựng lại** (tái lập được), đảm bảo tính lặp lại của đồ án.

### 5.2 Hạn chế & hướng phát triển

- RMSLE riêng lấy từ điểm Kaggle của **một** lần nộp mỗi nhánh; có thể nộp thêm biến thể /
  *kiểm định chéo theo thời gian* để ước lượng σ ổn định hơn.
- Thử thêm mô hình (Prophet, N-BEATS, …) hoặc trọng số trộn thay đổi theo nhóm hàng.
- Phân tích sai số theo cửa hàng / nhóm hàng / ngày lễ để hiểu rõ điểm yếu của mô hình.

## Tài liệu tham khảo

1. Kaggle — *Store Sales: Time Series Forecasting*. https://www.kaggle.com/competitions/store-sales-time-series-forecasting
2. Ansari et al. — *Chronos: Learning the Language of Time Series*, Amazon Science.
3. Herzen et al. — *darts: User-Friendly Modern ML for Time Series*, JMLR, 2022.
4. Ke et al. — *LightGBM: A Highly Efficient Gradient Boosting Decision Tree*, NeurIPS, 2017.
5. Chen & Guestrin — *XGBoost: A Scalable Tree Boosting System*, KDD, 2016.
6. Prokhorenkova et al. — *CatBoost: unbiased boosting with categorical features*, NeurIPS, 2018.
7. Chen et al. — *TSMixer: An All-MLP Architecture for Time Series Forecasting*, 2023.